In [1]:
#Loading all the necessary modules for the language model and chat model
from langchain_ollama import ChatOllama, OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_classic.chains import LLMChain, SequentialChain

chat = ChatOllama(model="mistral")


Language Model - raw LLM-- takes a string in, returns a string


In [2]:
llm = OllamaLLM(model="mistral", temperature=0.2, max_tokens=512)
response = llm.invoke("When did OpenAI models attack HuggingFace in one sentence, accuracy is of importance")
print(response)

 There was no reported instance where OpenAI models attacked HuggingFace. Both are organizations focused on AI research and development, not engaged in hostile actions against each other.


Chat Model- Layer over language model, built for multi-turn convesation with role-tagged messages; not flat messages.

In [3]:
chat = ChatOllama(model="mistral", temperature=0.2, max_tokens=512)
response = chat.invoke("When did OpenAI models attack HuggingFace in one sentence, accuracy is of importance")
print(response)


content=' There was no reported incident where OpenAI models attacked HuggingFace. Both are organizations that develop AI technologies and have collaborated on several projects, not competitors engaged in conflict.' additional_kwargs={} response_metadata={'model': 'mistral', 'created_at': '2026-07-27T16:29:13.2460995Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5832910400, 'load_duration': 122763600, 'prompt_eval_count': 21, 'prompt_eval_duration': 195380000, 'eval_count': 36, 'eval_duration': 5490987000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'} id='lc_run--019fa468-9dbb-7f92-82d8-57d50bc419b9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 21, 'output_tokens': 36, 'total_tokens': 57}


Chat message - role-tagged objects; what the above model consumes and produces  


In [6]:
messages = [
    SystemMessage(content="You are a an expert scientist that explains complex topics in simple terms."),
    HumanMessage(content="Explain the theory of relativity in simple terms."),
    AIMessage(content="The theory of relativity, developed by Albert Einstein, is a fundamental concept in physics that describes how space and time are interconnected. It consists of two main parts: special relativity and general relativity. Special relativity states that the laws of physics are the same for all observers, regardless of their relative motion, and it introduces the idea that time and space can stretch or contract depending on how fast you are moving. General relativity expands on this by explaining how gravity is not just a force between masses, but rather a curvature of space-time caused by mass and energy. In simple terms, it means that massive objects like planets and stars bend the fabric of space-time around them, which affects how objects move and how time passes."),
    HumanMessage(content="Can you provide a simple analogy to help me understand it better?"),  
]
response = chat.invoke(messages)
print(response)
type(response)  # This will show the type of the response object

content=" Sure! Here's an analogy to help explain the theory of relativity:\n\nImagine space-time is like a rubber sheet stretched out flat. Now, if you place a heavy ball (like a planet) on this sheet, it will cause the sheet to curve around the ball. This curvature affects how other objects move on the sheet. For example, if you roll a marble near the ball, it will follow a curved path instead of moving in a straight line.\n\nThis is similar to what happens with gravity according to general relativity. Massive objects like planets and stars bend space-time around them, causing other objects to move along curved paths. This bending of space-time is what we perceive as gravity.\n\nIn special relativity, the analogy is a bit different. Imagine you are on a train moving at high speed relative to someone standing still on the platform. To you, everything inside the train appears normal, but to the person on the platform, the train seems to be moving and time inside the train seems to slow

langchain_core.messages.ai.AIMessage

Prompt templates - reusable!

In [9]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates English to French."),
    ("human", "Explain {topic} in simple terms."),
])
prompt = template.format_prompt(topic="the theory of relativity")
print(prompt.to_messages())


[SystemMessage(content='You are a helpful assistant that translates English to French.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain the theory of relativity in simple terms.', additional_kwargs={}, response_metadata={})]


Output parsers - model's raw text response is coerced into structured type to use with code directly

In [10]:
#PydanticOutputParser here returns a RAGAnswer- I need to read both in detail
parser = StrOutputParser()
chain = template | chat | parser
result = chain.invoke({"topic": "the theory of relativity"})
print(type(result))

<class 'langchain_core.messages.base.TextAccessor'>


Puting these steps together, we create chains as shown above. 
Documents: standard unit for a piece of retrievable text plus its metadata

In [13]:
doc = Document(
    page_content="The theory of relativity, developed by Albert Einstein, is a fundamental concept in physics that describes how space and time are interconnected. It consists of two main parts: special relativity and general relativity. Special relativity states that the laws of physics are the same for all observers, regardless of their relative motion, and it introduces the idea that time and space can stretch or contract depending on how fast you are moving. General relativity expands on this by explaining how gravity is not just a force between masses, but rather a curvature of space-time caused by mass and energy. In simple terms, it means that massive objects like planets and stars bend the fabric of space-time around them, which affects how objects move and how time passes.",
    metadata={"source": "https://en.wikipedia.org/wiki/Theory_of_relativity"}
)


Agents-chain runs a fixed sequence each time it runs

In [15]:
# from langchain_classic.agents import initialize_agent, Tool, AgentType

# def calculator(expr: str) -> str:
#     return str(eval(expr))

# tools = [Tool(name="Calculator", func=calculator, description="Evaluates math expressions")]

# agent = initialize_agent(tools, chat, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
# result = agent.invoke("What is 47 * 12, and then explain the result in words?")
# print(result)

Using Chains

In [22]:
template = """Your job is to come up with a classic dish from 
    the area that the user suggests.
    {location}
"""
prompt_template = PromptTemplate(
    input_variables=["location"],
    template=template
)
# chain 1
location_chain = LLMChain(llm=llm, prompt=prompt_template, output_key="location_dish")

template = """Given the dish {location_dish}, come up with a simple recipe on how to make it at home.

    YOUR RESPONSE:
"""
recipe_template = PromptTemplate(
    input_variables=["location_dish"],
    template=template
)
# chain 2
recipe_chain = LLMChain(llm=llm, prompt=recipe_template, output_key="recipe")

template = """Given the recipe {recipe}, come up with a shopping list of ingredients needed to make it."""
shopping_list_template = PromptTemplate(
    input_variables=["recipe"],
    template=template
)
# chain 3
shopping_list_chain = LLMChain(llm=llm, prompt=shopping_list_template, output_key="shopping_list")

overall_chain = SequentialChain(
    chains=[location_chain, recipe_chain, shopping_list_chain],
    input_variables=["location"],
    output_variables=["location_dish", "recipe", "shopping_list"],
    verbose=True
)

overall_chain.invoke({"location": "Mutare"})

C:\Users\PC\AppData\Local\Temp\ipykernel_43056\3538697014.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  location_chain = LLMChain(llm=llm, prompt=prompt_template, output_key="location_dish")


{'location': 'Mutare',
 'location_dish': " One classic dish from Mutare, Zimbabwe, is Sadza na Nyama, which is a staple meal in Zimbabwean cuisine. Sadza is a type of cornmeal porridge, similar to polenta, and it's often served with Nyama (meat), usually beef or chicken, cooked in a variety of ways such as grilled, stewed, or fried. The dish is typically accompanied by vegetables like cabbage, carrots, and green beans, and it's often enjoyed with ground nuts or peanut butter sauce (Murondo) for added flavor.\n\nAnother popular dish from Mutare is Vetkoek, which are deep-fried dough balls filled with minced meat, onions, and chili sauce. These are a favorite street food in Zimbabwe.\n\nLastly, Samp and Beans is another traditional dish that originates from the Shona people of Zimbabwe, who are the predominant ethnic group in Mutare. Samp is a coarsely ground maize product, similar to hominy, cooked with beans until tender. It's often seasoned with onions, garlic, and salt, making for a 

In [36]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

In [37]:
pdf_files = [
    "./pdf_context/coctelectricity.pdf",
    "./pdf_context/chemASsyll.pdf"
]

# Load all documents into one list
all_docs = []
for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    all_docs.extend(docs)

In [20]:
print(all_docs[2].page_content[:500])  # Print the first 500 characters of the first document

Syllabus
Cambridge International  
AS & A Level  
Chemistry 9701
Use this syllabus for exams in 2025, 2026 and 2027 .
Exams are available in the June and November series.
Also available for examination in March 2025, 2026 and 2027 for India.
Version 1
For the purposes of screen readers, any mention in this document of Cambridge IGCSE 
refers to Cambridge International General Certificate of Secondary Education.


In [7]:
loader2 = WebBaseLoader("https://www.capetown.gov.za/Family%20and%20home/Residential-utility-services/Residential-electricity-services/the-cost-of-electricity")
web_docs = loader2.load()

In [38]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)

print(f"Number of chunks created: {len(chunks)}")
print(chunks[99].page_content[:1000])  # Preview first chunk

Number of chunks created: 100
School feedback:  ‘While studying Cambridge IGCSE and Cambridge International A Levels, students 
broaden their horizons through a global perspective and develop a lasting passion for learning.’
Feedback from: Zhai Xiaoning, Deputy Principal, The High School Affiliated to Renmin University of China
We are committed to making our documents accessible in accordance with the WCAG 2.1 Standard. We are always looking to improve 
the accessibility of our documents. If you find any problems or you think we are not meeting accessibility requirements, contact us at 
info@cambridgeinternational.org  with the subject heading: Digital accessibility. If you need this document in a different format, contact 
us and supply your name, email address and requirements and we will respond within 15 working days.
Cambridge Assessment International Education, The Triangle Building, Shaftesbury Road, Cambridge, CB2 8EA, United Kingdom
t : +44 (0)1223 553554  email : info@cambrid

In [39]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(model="nomic-embed-text")
texts = [doc.page_content for doc in chunks]
embedding_result = embedding_model.embed_documents(texts)
print(embedding_result[0][:5]) 


[0.037827473, 0.08263213, -0.2260274, -0.036463104, 0.06696225]


In [27]:
from langchain_chroma import Chroma

In [40]:
docsearch = Chroma.from_documents(chunks, embedding_model)
docs = docsearch.similarity_search("Langchain")
print(docs[0].page_content)

Cambridge International AS & A Level Chemistry 9701 syllabus for 2025, 2026 and 2027. Details of the assessment  
57
www.cambridgeinternational.org/alevelBack to contents page
Command words
Command words and their meanings help candidates know what is expected from them in the exam. The table 
below includes command words used in the assessment for this syllabus. The use of the command word will 
relate to the subject context. 
Command word What it means
Analyse examine in detail to show meaning, identify elements and the relationship between 
them
Calculate work out from given facts, figures or information
Compare identify/comment on similarities and/or differences
Consider review and respond to given information
Contrast identify/comment on differences
Deduce conclude from available information
Define give precise meaning
Demonstrate show how or give an example
Describe state the points of a topic / give characteristics and main features
Determine establish an answer using the inform

In [42]:
retriever = docsearch.as_retriever()
retriever.invoke("Langchain")

[Document(id='1de3d723-8b1a-468d-b63a-b359d074ef20', metadata={'keywords': 'Cambridge; syllabus; AS&A Level; Chemistry; 2025; 2026; 2027', 'total_pages': 98, 'source': './pdf_context/chemASsyll.pdf', 'creationdate': '2022-09-07T15:27:30+01:00', 'page_label': '57', 'producer': 'Adobe PDF Library 15.0', 'title': 'Cambridge International AS&A Level 9701 Chemistry syllabus for examination in 2025, 2026 and 2027', 'author': 'Document Production Team', 'creator': 'Adobe InDesign 16.1 (Windows)', 'moddate': '2022-09-07T15:30:08+01:00', 'page': 56, 'trapped': '/False'}, page_content='Cambridge International AS & A Level Chemistry 9701 syllabus for 2025, 2026 and 2027. Details of the assessment  \n57\nwww.cambridgeinternational.org/alevelBack to contents page\nCommand words\nCommand words and their meanings help candidates know what is expected from them in the exam. The table \nbelow includes command words used in the assessment for this syllabus. The use of the command word will \nrelate to t

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

store = InMemoryStore()
vectorstore = Chroma(embedding_function=embedding_model, collection_name="parent_child_demo")

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
retriever.add_documents(all_docs)

sub_docs = vectorstore.similarity_search("Langchain")   # small chunk matched
retrieved_docs = retriever.invoke("Langchain")           # full parent returned

In [48]:
print("--- sub_docs (small child chunk) ---")
print(len(sub_docs[0].page_content), "characters")
print(sub_docs[0].page_content)

print("\n--- retrieved_docs (full parent chunk) ---")
print(len(retrieved_docs[0].page_content), "characters")
print(retrieved_docs[0].page_content)


--- sub_docs (small child chunk) ---
368 characters
• Servicing informal settlements. 
• Servicing green litter bins and large   
 central business districts bins/   
 containers. 
• Animal carcass removals. 
• Unscheduled residential cleaning   
 where resources allow. 
While City-supplied customers have 
always contributed to City-Wide 
Cleaning via electricity purchases, this 
is a new charge for Eskom-supply area

--- retrieved_docs (full parent chunk) ---
1990 characters
LIFELINE ELECTRICITY: MORE RESIDENTS NOW QUALIFY!
Pensioners and social grant recipients 
now qualify for Lifeline Electricity up to 
R27 000 monthly household income! (up 
from R22 000). 
Indigent households qualify for lifeline 
up to R500 000 property value or R7 500 
monthly income.
This is SA’s widest qualifying criteria!
Lifeline customers using 600 units p/m still pay roughly the same as they did 
three years ago. 
Customers need to stay within the 450-unit monthly average over 12 months to 
remain on the L